# 🔎 Búsqueda de costo uniforme (UCS)

El algoritmo **Uniform Cost Search (UCS)** selecciona para expansión el nodo con el menor costo acumulado desde el nodo inicial. Para ello, utiliza una **cola de prioridad** que organiza los nodos de acuerdo con el costo de la ruta que conduce a cada uno de ellos. Es importante tener en cuenta que se muestran dos versiones de implementación de UCS, una sin controlar actualización de costos ni nodos obsoletos y otra, en la que este control si se realiza. El objetivo de mostrar ambas versiones es identificar que se puede obtener el mismo camino y el mismo costo en ambos casos, pero cuando no se controlan nodos obsoletos ni actualización de costos, se pueden realizar expansiones innecesarias de algunos nodos, lo cual conlleva al aumento del costo computacional.

### ⚙️ Funcionamiento del algoritmo

El costo acumulado de un nodo $n$, denotado por $g(n)$, se calcula sumando los costos de las acciones que conforman la ruta desde el nodo inicial hasta dicho nodo:

$
g(n) = \sum_{(u,v)\in \text{ruta}(n)} c(u,v),
$

donde $c(u,v)$ representa el costo de la acción que permite pasar del nodo $u$ al nodo $v$. La cola de prioridad garantiza que, en cada iteración, se seleccione primero el nodo con el menor costo acumulado. De esta manera, bajo las condiciones de optimalidad del algoritmo, UCS encuentra una **ruta de costo mínimo hacia la meta**.

### 🔄 Actualización de costos y control de nodos obsoletos

Un aspecto importante de la implementación es que un mismo nodo puede alcanzarse mediante diferentes rutas. Por tanto, es necesario:

- **Actualizar los costos:** si se encuentra una ruta hacia un nodo con un costo menor que el registrado previamente, se debe actualizar su costo.
- **Controlar los nodos obsoletos:** cuando un nodo se extrae de la cola de prioridad, es necesario verificar que el costo asociado corresponda al mejor costo conocido. Si existe una ruta más económica descubierta posteriormente, la entrada anterior se considera obsoleta y no debe generar una expansión innecesaria.

### 🌳 Representación del problema

En este ejemplo, tanto el grafo como el árbol de búsqueda se representan mediante diccionarios de Python. La siguiente figura ilustra ambas representaciones del mismo problema:

<center>
    <img src="../resources/figures/ucs.png"
         style="width: 600px; max-width: 100%; height: auto;">
    <p style="text-align: center;">
        <b>Figura 1.</b> Problema de búsqueda representado como grafo y como árbol de búsqueda.
    </p>
</center>

### 🧪 Variación del problema

Es posible modificar los costos de las aristas y la estructura del problema para observar cómo cambia el orden de expansión de los nodos y la ruta encontrada por UCS. Preste especial atención a los casos en los que un mismo nodo puede alcanzarse mediante diferentes rutas, pues permiten analizar el mecanismo de actualización de costos y el tratamiento de entradas obsoletas en la cola de prioridad.

In [102]:
import heapq
from collections import Counter


problema = {
    'A': [('B', 5), ('C', 1)],
    'B': [('D', 2), ('E', 5)],
    'C': [('B', 1), ('D', 6)],
    'D': [('G', 3)],
    'E': [('G', 1)],
    'G': []
}


def ucs_sin_actualizar(problema, inicio, meta):
    """
    Algoritmo UCS sin actualización de costos de nodos ni control de nodos obsoletos.
    
    Args:
        problema: árbol o grafo de búsqueda.
        inicio: nodo inicial.
        meta: nodo meta.
    Returns:
        camino: ruta óptima a la meta.
        costo_actual: el costo acumulado óptimo.
        expansiones: diccionario con el número de veces que se expande cada nodo.
        costos: listado de costos acumulados de los nodos.
        None: cuando no hay solución.
    Raises:
        None.
    """
    priorityQueue = [(0, inicio, [inicio])] # Se crea la cola de prioridad con el costo del nodo inicial, el nodo inicial y su ruta
    costos = {inicio: 0} # Diccionario para actualizar costos acumulados
    expansiones = Counter()# Permite contar el número de veces que aparece un nodo

    while priorityQueue:
        costo_actual, nodo_actual, camino = heapq.heappop(priorityQueue) # Se extrae el costo, el nodo y la ruta hacia él

        if nodo_actual == meta:
            return camino, costo_actual, expansiones, costos

        expansiones[nodo_actual] += 1 # Se incrementa cada vez que un nodo es expandido

        for sucesor, costo_accion in problema[nodo_actual]:
            nuevo_costo = costo_actual + costo_accion
            costos[sucesor] = nuevo_costo # Se actualiza el costo sin importar si es menor o mayor
            heapq.heappush(
                priorityQueue,
                (nuevo_costo, sucesor, camino + [sucesor])
            )

    return None


def ucs_actualizando(problema, inicio, meta):
    """
    Algoritmo UCS con actualización de costos de nodos y control de nodos obsoletos.
    
    Args:
        problema: árbol o grafo de búsqueda.
        inicio: nodo inicial.
        meta: nodo meta.
    Returns:
        camino: ruta óptima a la meta.
        costo_actual: el costo acumulado óptimo.
        expansiones: diccionario con el número de veces que se expande cada nodo.
        costos: listado de costos acumulados de los nodos.
        None: cuando no hay solución.
    Raises:
        None.
    """
    priorityQueue = [(0, inicio, [inicio])]# Se crea la cola de prioridad con el costo del nodo inicial, el nodo inicial y su ruta
    costos = {inicio: 0}# Diccionario para actualizar costos acumulados
    expansiones = Counter()# Permite contar el número de veces que aparece un nodo

    while priorityQueue:
        costo_actual, nodo_actual, camino = heapq.heappop(priorityQueue)# Se extrae el costo, el nodo y la ruta hacia él

        # Descartar entradas obsoletas
        if costo_actual > costos[nodo_actual]:
            continue

        if nodo_actual == meta:
            return camino, costo_actual, expansiones, costos

        expansiones[nodo_actual] += 1# Se incrementa cada vez que un nodo es expandido

        for sucesor, costo_accion in problema[nodo_actual]:
            nuevo_costo = costo_actual + costo_accion

            if (sucesor not in costos or
                    nuevo_costo < costos[sucesor]): # Se actualizan costos de nodos cuando hay rutas mejores hacia ellos

                costos[sucesor] = nuevo_costo

                heapq.heappush(
                    priorityQueue,
                    (nuevo_costo, sucesor, camino + [sucesor])
                )

    return None


for nombre, algoritmo in [
    ("Sin actualizar costos", ucs_sin_actualizar),
    ("Actualizando costos", ucs_actualizando)
]:
    camino, costo, expansiones, costos = algoritmo(problema, 'A', 'G')

    print(f"\n{nombre}")
    print("Camino:", camino)
    print("Costo:", costo)
    print("Costos de los nodos:", costos)
    print("Expansiones por estado:", dict(expansiones))
    print("Total de expansiones:", sum(expansiones.values()))


Sin actualizar costos
Camino: ['A', 'C', 'B', 'D', 'G']
Costo: 7
Costos de los nodos: {'A': 0, 'B': 2, 'C': 1, 'D': 7, 'E': 10, 'G': 8}
Expansiones por estado: {'A': 1, 'C': 1, 'B': 2, 'D': 3, 'E': 1}
Total de expansiones: 8

Actualizando costos
Camino: ['A', 'C', 'B', 'D', 'G']
Costo: 7
Costos de los nodos: {'A': 0, 'B': 2, 'C': 1, 'D': 4, 'E': 7, 'G': 7}
Expansiones por estado: {'A': 1, 'C': 1, 'B': 1, 'D': 1, 'E': 1}
Total de expansiones: 5
